<a href="https://colab.research.google.com/github/eogks1235-byte/DEEP_Learning-feat.Jake-Oh/blob/main/ml21_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM(Large Language Model, 거대 언어 모델)

* 시퀀스-시퀀스 작업(Sequence-to-Sequence)
  * 시퀀스 데이터(sequence data)를 입력으로 받아서 시퀀스 데이터를 출력하는 작업.
  * 자연어 처리(NLP, Natural Language Processing) 분야에서 요약, 변역 등의 작업.
  * 두 개 (순환) 신경망 연결한 인코더-디코더(encoder-decoder) 구조가 널리 사용됨.
* 어텐션 메커니즘(Attention mechanism)
  * 인코더-디코더 구조에서 사용된 순환 신경망(RNN)의 성능을 향상시키기 위해서 고안.
  * 기존에는 인코더의 마지막 타입 스텝에서 출력한 은닉 상태만을 사용해서 디코더가 새로운 텍스트를 생성.
  * 어텐션 메커니즘은 모든 타임 스텝에서 인코더가 출력한 은닉 상태를 디코더가 참조할 수 있도록 고안.
  * 디코더가 새로운 토큰을 생성할 때 인코더가 처리한 토큰들 중에서 어떤 토큰에 더 주의(attention)를 기울일지를 결정. 입력 토큰들마다 디코더가 중요도를 다르게 부여.
* 트랜스포머 모델(Transformer model)
  * 어텐션 메커니즘을 기반으로 해서 인코더-디코더 구조에서 순환층을 제거.
  * 인코더에서 한 번에 하나의 토큰을 처리하지 않고 입력 텍스트 전체를 한 번에 처리.
  * 핵심 구성 요소
    * 멀티 헤드 어텐션(multi-head attention)
    * 층 정규화(layer normalization)
    * 잔차 연결(residual connection)
    * 피드 포워드 네트워크(feed-foeward network) (conv2D-dense 입력-입력-입력 같은느낌)


<img src="https://camo.githubusercontent.com/e14088bbac1a33963fc8f590e902161866e0d571f4f8a4e3e8571512bc55b148/68747470733a2f2f7062732e7477696d672e636f6d2f6d656469612f474a673351744d58774141765554343f666f726d61743d6a7067266e616d653d6c61726765"
  alt="LLM 발전 과정" />

* 오른쪽 회색 가지
  * Transformer 모델을 사용하지 않은 알고리즘.
  * 워드 임베딩 벡터를 만드는 알고리즘.
* Encoder 기반 모델
  * 텍스트(긍정/부정) 분류
  * 객체명 인식 - 텍스트에서 사람 이름, 지역 이름, 회사 이름 등의 고유 명사를 식별.
  * BERT, RoBERTa, ALBERT, ...
* Encoder-Decoder 기반 모델
  * 문서 요약, 번역, 질문-답변
  * T5, BART, ...
* Decoder 기반 모델
  * 텍스트 생성 - 챗봇, 질문-답변, 요약, 번역, ...
  * 디코더
    * 이전까지 생성한 텍스트를 입력받아서 다음 토큰을 예측하는 방식.
    * 인코더로부터 입력이 없으면 디코더는 아무것도 생성할 수 없음.
    * 이전에 생성한 텍스트인 것처럼 어떤 텍스트를 입력해주면 인코더 도움 없이 다음 토큰을 예측할 수 있다.
    * 프롬프트(prompt): 이전에 생성된 텍스트인 것처럼 전달하는 초기 텍스트.
  * GPT-4, GPT-5, LLaMA, Claude, ...
  * 현재 가장 활발히 연구되는 LLM분야.
    

# BART 모델을 사용한 문서 요약

Hugging Face - 오픈 소스 LLM 모델(Transformer 기반)들 제공. Transformer 모델들을 사용할 수 있는 패키지 제공.

In [111]:
# import transformers # Hugging Face 에서 제공하는 Transformer 모델들을 사용할 수 있는 패키지


In [112]:
# transformers.__version__

* 2026/03/23  현재 transformers 패키지의 가장 최신 안정(stable)적인 버전은 5.3.0
  * Google Colab에 설치된 버전 5.0.0
* transformers 패키지 5.0.0 버전부터는 pipeline 함수에서 "summarization" task(작업)이 삭제됨.
* BART 모델 소개를 위해서 이전 버전의 transformer 패키지를 Colab에 설치.
  * 4.x 버전에서 가장 마지막 안정(stable) 버전은 4.57.6  

In [76]:
# 코드 셀에서 Linux 명령어 실행하기
#                특정버전 == 이거 필요
# pip: Python 패키지 관리자 명령어. 패키지 설치/삭제/업데이트.
# pip install [옵션] 패키지이름[==특정버전]
# import 하고 하면 런타임 오류남
!pip install transformers==4.57.6

In [77]:
import transformers # Hugging Face 에서 제공하는 Transformer 모델들을 사용할 수 있는 패키지


In [78]:
transformers.__version__

'4.57.6'

In [79]:
# 깃허브에 .ipynb 파일을 업로드할 때 다운로드 상태(진행바) 표시줄때문에 오류 발생
# -> 깃허브에 정상적으로 업로드되게 하기 위해서
transformers.utils.logging.disable_progress_bar()

In [80]:
# Hugging Face에서 오픈 소스로 공개된 모델을 다운로드해서 로컬에서 실행할 수 있도록 함.
# pipeline() 함수 호출 - Pipeline 객체 생성
# pipeline() 함수의 model 파라미터를 설정하지 않으면 "sshleifer/distilbart-cnn-12-6" 모델을 기본값으로 다운로드
# sshleifer/distilbart-cnn-12-6 핵심
# pipeline() 함수의 device 파라미터: device=-1(기본값, CPU 사용), device=0(GPU 계산)

In [81]:
pipe = transformers.pipeline(task="summarization")

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [82]:
print(type(pipe))

<class 'transformers.pipelines.text2text_generation.SummarizationPipeline'>


In [83]:
# ''' 덩어리 이해 ''' 덩어리 이해 '''
sample_text ='''
 large language model (LLM) is a computational model trained on a vast amount of data, designed for natural language processing tasks, especially language generation.[1][2] The largest and most capable LLMs are generative pre-trained transformers (GPTs) that provide the core capabilities of modern chatbots. LLMs can be fine-tuned for specific tasks or guided by prompt engineering.[3] These models acquire predictive power regarding syntax, semantics, and ontologies[4] inherent in human language corpora, but they also inherit inaccuracies and biases present in the data they are trained on.[5]

LLMs consist of billions to trillions of parameters and operate as general-purpose sequence models, generating, summarizing, translating, and reasoning over text. LLMs represent a significant new technology in their ability to generalize across tasks with minimal task-specific supervision, enabling capabilities like conversational agents, code generation, knowledge retrieval, and automated reasoning that previously required bespoke systems.[6]

LLMs evolved from earlier statistical and recurrent neural network approaches to language modeling. The transformer architecture, introduced in 2017, replaced recurrence with self-attention, allowing efficient parallelization, longer context handling, and scalable training on unprecedented data volumes.[7] This innovation enabled models like GPT, BERT, and their successors, which demonstrated emergent behaviors at scale, such as few-shot learning and compositional reasoning.[8]

Reinforcement learning, particularly policy gradient algorithms, has been adapted to fine-tune LLMs for desired behaviors beyond raw next-token prediction.[9] Reinforcement learning from human feedback (RLHF) applies these methods to optimize a policy, the LLM's output distribution, against reward signals derived from human or automated preference judgments.[10] This has been critical for aligning model outputs with user expectations, improving factuality, reducing harmful responses, and enhancing task performance.

Benchmark evaluations for LLMs have evolved from narrow linguistic assessments toward comprehensive, multi-task evaluations measuring reasoning, factual accuracy, alignment, and safety.[11][12] Hill climbing, iteratively optimizing models against benchmarks, has emerged as a dominant strategy, producing rapid incremental performance gains but raising concerns of overfitting to benchmarks rather than achieving genuine generalization or robust capability improvements.[13]
'''

In [84]:
# Pipeline 객체를 함수처럼 호출 - sample_text를 요약
result=pipe(sample_text)

In [85]:
print(result)

[{'summary_text': ' The largest and most capable LLMs are generative pre-trained transformers (GPTs) that provide the core capabilities of modern chatbots . LLMs consist of billions to trillions of parameters and operate as general-purpose sequence models, generating, summarizing, translating, and reasoning over text . They can be fine-tuned for specific tasks or guided by prompt engineering .'}]


pipe() 함수의 리턴 값은 dict를 아이템으로 갖는 list.

list가 가지고 있는 dict 개수는 1개.

dict는 (key-value) 아이템을 1개만 갖고 있음.

In [86]:
len(result)

1

In [87]:
result[0] #> list의 첫번째 원소 - dict(key-value 아이템 1개)

{'summary_text': ' The largest and most capable LLMs are generative pre-trained transformers (GPTs) that provide the core capabilities of modern chatbots . LLMs consist of billions to trillions of parameters and operate as general-purpose sequence models, generating, summarizing, translating, and reasoning over text . They can be fine-tuned for specific tasks or guided by prompt engineering .'}

In [88]:
result[0]['summary_text']

' The largest and most capable LLMs are generative pre-trained transformers (GPTs) that provide the core capabilities of modern chatbots . LLMs consist of billions to trillions of parameters and operate as general-purpose sequence models, generating, summarizing, translating, and reasoning over text . They can be fine-tuned for specific tasks or guided by prompt engineering .'

In [89]:
sample_text='''
The largest and most capable LLMs are generative pre-trained transformers (GPTs) that provide the core capabilities of modern chatbots . LLMs consist of billions to trillions of parameters and operate as general-purpose sequence models, generating, summarizing, translating, and reasoning over text . They can be fine-tuned for specific tasks or guided by prompt engineering .
'''

In [90]:
result=pipe(sample_text)

Your max_length is set to 142, but your input_length is only 81. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=40)


In [91]:
result[0]['summary_text']

' The largest and most capable LLMs are generative pre-trained transformers (GPTs) that provide the core capabilities of modern chatbots . LLMs consist of billions to trillions of parameters and operate as general-purpose sequence models . They can be fine-tuned for specific tasks or guided by prompt engineering .'

위에서 사용한 LLM 모델 "sshleifer/distilbart-cnn-12-6"은 영어만 훈련된 모델. 한글 텍스트를 요약하려고 하면 에러가 발생. 한글 요약 작업을 하기 위해서는 한글로 훈련된 모델을 다운로드해서 실행해야 됨.

## KoBART 모델을 사용한 한글 문서 요약

In [92]:
pipe_kor=transformers.pipeline(task='summarization',              # task=작업내용
                              model='EbanLee/kobart-summary-v3',  # model=다운로드받을 모델명
                               device=-1)                         # divice=GPU 사용 여부 -1 안씀 / 0 씀

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Device set to use cpu


In [93]:
sample_text_kor= '''
독일 발도르프에 본사를 두고 있는 SAP는 글로벌지원 최고책임자인 거드 오스왈드(Gehard Oswald)를 내세워 인재양성의 협약을 체결했다. 선발된 SW인재는 국내뿐 아니라 SAP의 글로벌 R&D연구센터와 업무현장에 배치될 예정이어서 한국 SW인재의 해외진출의 문이 활짝 열리게 됐다는 평가이다.[1]

SAP가 채용하는 인력은 SAP코리아가 지원해 최근 판교에 설립한 디코리아(D Korea)재단이 맡는다. 단국대학교도 인력양성에 참여한다. 경기도와 SAP는 하반기에 우선 200명의 SW인력을 뽑아 교육을 마친 후 현업에 배치할 예정이다. 나머지 인력은 2018년까지 매년 200명씩 순차적으로 선발해 채용하게 된다.[2]

SAP가 한국에서 SW인력을 대폭 충원키로 한 것은 지난해 주력 제품으로 내놓은 SAP HANA 플랫폼의 성장세에 따른 것이다. SAP HANA 플랫폼의 급격한 성장이 한국 인재 채용에 결정적 배경이 됐다. SAP HANA는 SAP코리아 R&D센터가 개발하고 있는 제품이다.[3]

현재 오라클에서 업계 최초 비즈니스 애널리틱스 머신인 오라클 엑사리틱스 인메모리 머신(Oracle Exalytics In-Memory Machine)을 내놓으면서 분석 솔루션 시장에서 입지를 강화시키고 있는데 이 오라클 엑사리틱스 인메모리 머신은 데이터 분석, 비즈니스 인텔리전스 및 애플리케이션의 모델링 및 플래닝의 생산성과 성능을 향상시키고, IT 인프라의 비용 및 복잡성을 제거하기 위해 고성능 하드웨어에 엔지니어드하여 업계 최상의 애널리틱스 및 인메모리 소프트웨어를 제공한다고 밝혀 놓은 상태이다.[4]

또한 IBM에서는 스마터 데이터센터 전략으로 현재 데이터센터 상태뿐만 아니라 데이터센터 수명 전체를 통틀어 진단하는 IBM의 특허기술인 데이터센터 애널리틱스 툴을 기반으로 하는 서비스 툴을 내놓았다.

IBM 스마터 데이터센터의 주요 서비스는 ▲ 데이터센터 성장한계 분석(Physical Infrastructure Threshold Analysis) ▲ 장애 및 재해시 복원력 분석(Resiliency Rationalization Analysis) ▲ 투자비용 분석(Cash flow Anlysis) ▲ 시스템 간 논리적 연결 분석(Analytics for Logical Dependency Mapping) 등이다.

'데이터센터 성장한계 분석' 툴은 기존 데이터센터가 언제까지 사용가능한지를 판단해 고객이 차세대 데이터센터 구축 시기 대해 미리 파악하고 향후 계획 수립에 미리 대비할 수 있도록 돕는다.
‘장애 및 재해시 복원력 분석 툴’은 데이터센터를 통합할 때 비지니스 연속성을 확보할 수 있는 사이트의 개수를 판단해 재해 복구 능력을 개선한다.
’투자비용 분석 툴’은 고객이 데이터센터를 구축 또는 임대할 때 투자되는 비용을 분석해 투자 의사결정을 돕는다.
’시스템간 논리적 연결 분석’은 시스템간 논리적 의존성을 분석해 서버 이전 시 특정 업무에 사용하는 업무 시스템을 분리해 내는 툴이다.
한편 HP에서는 `HP 비즈니스 애널리틱스 서비스'를 출시했다고 밝혔다.


'''

In [94]:
result=pipe_kor(sample_text_kor)

In [95]:
print(result)

[{'summary_text': '독일 발도르프에 본사를 두고 있는 SAP는 인재양성의 협약을 체결했다. 선발된 SW인재는 국내뿐 아니라 SAP의 글로벌 R&D연구센터와 업무현장에 배치될 예정이다. SAP HANA 플랫폼의 급격한 성장이 한국 인재 채용에 결정적 배경이 되었다. IBM에서는 데이터센터 수명 전체를 통틀어 진단하는 IBM의 특허기술인 데이터센터 애널리틱스 툴을 기반으로 하는 서비스 툴을 내놓았다. IBM 스마터 데이터센터의 주요 서비스는 데이터센터 성장한계 분석, 장애 및 재해시 복원력 분석, 투자비용 분석, 시스템 간 논리적 연결 분석 등이 있다.'}]


In [96]:
result[0]['summary_text']

'독일 발도르프에 본사를 두고 있는 SAP는 인재양성의 협약을 체결했다. 선발된 SW인재는 국내뿐 아니라 SAP의 글로벌 R&D연구센터와 업무현장에 배치될 예정이다. SAP HANA 플랫폼의 급격한 성장이 한국 인재 채용에 결정적 배경이 되었다. IBM에서는 데이터센터 수명 전체를 통틀어 진단하는 IBM의 특허기술인 데이터센터 애널리틱스 툴을 기반으로 하는 서비스 툴을 내놓았다. IBM 스마터 데이터센터의 주요 서비스는 데이터센터 성장한계 분석, 장애 및 재해시 복원력 분석, 투자비용 분석, 시스템 간 논리적 연결 분석 등이 있다.'

In [97]:
sample_text_korr='''이번에 출시한 재무분석 서비스는 내부 데이터베이스(DB)와 같은 정형데이터 처리에 익숙한 재무 그룹에게 검색엔진을 지원해 비정형 데이터에서도 핵심 정보를 찾을 수 있게 한다. 또 이를 바탕으로 정형ㆍ비정형 데이터를 통합해 개선된 의사 결정 프로세스를 도출하도록 지원한다.

한국HP는 이번 서비스로 도출된 데이터를 분석할 수 있는 전문 분석가를 제공하고, 고객사와 협의해 특정 활동, 시스템, 시간대별 서비스를 지원할 예정이다.

이러한 서비스는 기업이 필요로 하는 재무 와 비지니스 데이터 분석에 대한 요구사항을 지원해 줄 수 있는 서비스로, 새로운 분석 서비스를 제공 할 것이라고 업계 전문가들은 말하고 있다.

2012년 기준, 상위 6대 벤더사인 SAP, 오라클, IBM, HP, 마이크로소프트, SAS, 테라데이타가 전 세계 매출의 64%를 차지하고 있다. 2012년 기준 대한민국 시장은 전년 대비 8.6% 성장한 3천120억원(소프트웨어 라이선스 기준) 규모이다.[5]

이외에도 포털 사이트 중심으로 자체적으로 분석 툴을 내놓고 있는데 스트래티지 애널리틱스(Strategy Analytics), 액센츄어 애널리틱스(Accenture Analytics), 구글 애널리틱스, 소셜 애널리틱스, 웹 애널리틱스, 모바일 애널리틱스 등 새롭게 나오는 비즈니스 애널리틱스 머신들로 종류도 다양하게 나와 있다.

이처럼 최근 클라우드 컴퓨팅과 함께 빅데이터가 IT업계뿐 아니라 전 비즈니스 업계에 화두가 되었다. 빅데이터(Big Data)로 미래를 예측할 수 있는 실례가 발표되면서 최근 빅데이터를 활용한 흥미로운 연구결과들이 공개되고 있다.

세계 최대의 검색업체인 구글은 최근 몇 년 동안 독감예측 자료를 공개하였는데 구글 사용자들의 검색 통계(빅데이터)를 기반으로 시간별, 지역별 독감 유행 정보를 제공한 것인데 결과는 미국 질병통제센터의 발표 내용과 거의 일치하였다고 한다.

최근에는 트위터 댓글들을 분석하면 구글이나 미국 질병통제센터보다도 더 빨리 독감 확산 경로나 특정인이 언제 독감에 걸릴지도 예측할 수 있다는 연구결과도 소개가 됐는데 빅데이터를 활용한 미래 예측의 사례들로 평가되고 있다.

빅데이터를 활용한 또 다른 흥미로운 사례는 2012년 4월 미국의 미디어 업계에서 일어났다. 시카고의 한 지역 신문사가 기자 20여 명을 해고하고 ‘저너틱’이라는 회사에 기사를 아웃소싱했다고 한다. 그런데 ‘저너틱’은 빅데이터 기술 알고리즘과 로봇을 활용해 기사를 작성하는 회사였다. 이렇게 로봇이 기사를 작성할 수 있도록 사람의 업무를 대체되는 시대가 되었다는 의미이다.[6]
'''

In [98]:
result=pipe_kor(sample_text_korr)

In [99]:
print(result)

[{'summary_text': '정형데이터 처리에 익숙한 재무 그룹에게 검색엔진을 지원해 비정형 데이터에서도 핵심 정보를 찾을 수 있게 하고 이를 바탕으로 정형ᆞ비정형 데이터를 통합해 개선된 의사 결정 프로세스를 도출하도록 지원한다.'}]


In [100]:
result[0]['summary_text']

'정형데이터 처리에 익숙한 재무 그룹에게 검색엔진을 지원해 비정형 데이터에서도 핵심 정보를 찾을 수 있게 하고 이를 바탕으로 정형ᆞ비정형 데이터를 통합해 개선된 의사 결정 프로세스를 도출하도록 지원한다.'

# BART 모델 동작 원리

* encoder-decoder 모델
* 텍스트 입력 --> encode() --> LLM 모델 --> decode() --> 텍스트 출력
* pipeline() 함수는 LLM 모델과 함께 텍스트를 토큰화하는 tokenizer 객체를 함께 다운로드.
* tokenizer 객체가 텍스트 인코딩과 디코딩을 수행하는 메서드를 가지고 있음.

In [101]:
tokenizer = pipe_kor.tokenizer
print(type(tokenizer))

<class 'transformers.tokenization_utils_fast.PreTrainedTokenizerFast'>


In [102]:
# tokenize() 메서드: 텍스트 -> 토큰들의 리스트
tokens = tokenizer.tokenize('혼자 공부하는 머신러닝 + 딥러닝')
print(tokens)

['▁혼자', '▁공부', '하는', '▁머', '신', '러', '닝', '▁+', '▁', '딥', '러', '닝']


In [103]:
# convert_tokens_to_ids(): 토큰들의 리스트를 인코딩된 정수(인덱스)들의 리스트로 변환
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids) # 글자(토큰) > 숫자(인덱스?)

[16814, 16962, 14049, 14771, 11467, 10277, 9747, 23020, 1700, 10021, 10277, 9747]


In [104]:
# convert_ids_to_token(): 토큰 아이디(인덱스)들의 리스트를 해당하는 토큰들의 리스트로 변환
result = tokenizer.convert_ids_to_tokens(ids)
print(result) # 숫자(인덱스?) > 글자(토큰)

['▁혼자', '▁공부', '하는', '▁머', '신', '러', '닝', '▁+', '▁', '딥', '러', '닝']


In [105]:
# encode(): tokenize + convert_tokens_to_ids
encoded= tokenizer.encode('KoBART 모델을 사용한 문서 요약')
print(encoded)
# encode() 메서드는 텍스트 시작(0)과 끝(1)을 나타내는 아이디가 포함.

[0, 14572, 310, 265, 264, 281, 283, 24224, 21032, 26052, 26200, 1]


In [106]:
tokenizer.convert_ids_to_tokens(encoded)
# 0, 1: <s>, </s>

['<s>', '▁K', 'o', 'B', 'A', 'R', 'T', '▁모델을', '▁사용한', '▁문서', '▁요약', '</s>']

In [107]:
# decode(): 토큰 아이디들의 리스트를 텍스트로 변환
decoded= tokenizer.decode(encoded)
print(decoded)
# 디코딩된 텍스트에는 시작<s>과 끝</s>을 나타내는 특수 기호가 포함.

<s> KoBART 모델을 사용한 문서 요약</s>


In [108]:
tokenizer.convert_ids_to_tokens(encoded)[1:-1]

['▁K', 'o', 'B', 'A', 'R', 'T', '▁모델을', '▁사용한', '▁문서', '▁요약']

In [109]:
encoded[1:-1] # 인코딩된 아이디 리스트의 첫번째 원소는 항상 0 - 텍스트 시작을 의미

[14572, 310, 265, 264, 281, 283, 24224, 21032, 26052, 26200]

In [110]:
tokenizer.decode(encoded[1:-1]) # 인코딩된 아이디 리스트의 마지막 원소는 항상 1 - 텍스트 마지막을 의미

'KoBART 모델을 사용한 문서 요약'